# Differential Privacy Researcher Tutorial

This notebook is designed for developers and researchers prototyping DP-RAG mechanisms in one place.

What you will do:
- Build a DP corpus from a single `PrivacyConfig` (no epsilon mismatch).
- Run a multi-round agent and inspect an accumulated-RDP budget snapshot.
- Prototype and register a custom `DPMechanismPlugin` in-notebook.
- Compare mechanisms (`gaussian`, `laplace`, custom) with the same corpus/query.
- Verify ergonomic behavior: budget is only debited after successful retrieval.

In [1]:
from __future__ import annotations

from pathlib import Path
import hashlib
import random
import socket
import subprocess
import sys
import time
from types import SimpleNamespace

import httpx

# Make the local workspace importable when running this notebook directly.
cwd = Path.cwd().resolve()
repo_root = next(
    (p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists() and (p / "securerag").is_dir()),
    None,
 )
if repo_root is None:
    raise RuntimeError("Could not locate repository root containing pyproject.toml and securerag/")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from securerag import DPMechanismPlugin, PrivacyConfig, PrivacyProtocol, SecureRAGAgent
from securerag.backend_client import create_backend
from securerag.budget import BudgetManager
from securerag.corpus import CorpusBuilder
from securerag.errors import BackendError
from securerag.models import RawDocument
from securerag.retriever import PrivacyRetriever
from securerag.llm import OllamaLLM

import securerag.retrievers  # ensure protocol retrievers are registered


def free_port() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return int(s.getsockname()[1])


def launch_sim_server(port: int) -> subprocess.Popen:
    return subprocess.Popen(
        [
            sys.executable,
            "-m",
            "uvicorn",
            "securerag.sim_server:app",
            "--host",
            "127.0.0.1",
            "--port",
            str(port),
        ],
        cwd=str(repo_root),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )


def wait_for_health(base_url: str, proc: subprocess.Popen | None = None, timeout: float = 10.0) -> None:
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc is not None and proc.poll() is not None:
            stderr_out = ""
            if proc.stderr is not None:
                stderr_out = proc.stderr.read().strip()
            raise RuntimeError(
                f"sim_server exited early with code {proc.returncode}. stderr:\n{stderr_out}"
            )
        try:
            r = httpx.get(f"{base_url}/health", timeout=0.4)
            if r.status_code == 200:
                return
        except Exception:
            pass
        time.sleep(0.1)
    raise RuntimeError(f"sim_server did not become healthy at {base_url}")


def print_snapshot(snap: dict) -> None:
    print("spent:", round(snap["spent"], 6))
    print("remaining:", round(snap["remaining"], 6))
    print("rounds:", snap["rounds"])
    print("mechanism:", snap.get("mechanism", "unknown"))
    print("ledger:", [(r, round(eps, 6)) for r, eps in snap["ledger"]])

## 1) Budget Intuition: Accumulated RDP vs Naive Per-Round Sum

The current budget manager composes in RDP and converts to $(\epsilon, \delta)$ after accumulation.

This cell prints how epsilon grows across rounds for several noise levels.

In [2]:
for sigma in [0.8, 1.0, 1.5]:
    cfg = PrivacyConfig(protocol=PrivacyProtocol.DIFF_PRIVACY, epsilon=100.0, delta=1e-5, noise_std=sigma)
    b = BudgetManager(cfg)
    row = []
    for _ in range(4):
        b.consume(sigma=sigma)
        row.append(round(b.spent, 4))
    print(f"sigma={sigma}: epsilon trajectory by round -> {row}")

sigma=0.8: epsilon trajectory by round -> [6.9626, 10.0876, 13.2126, 16.3376]
sigma=1.0: epsilon trajectory by round -> [5.6447, 7.8376, 9.8376, 11.8376]
sigma=1.5: epsilon trajectory by round -> [3.4225, 5.2003, 6.5043, 7.3932]


## 2) Build a DP Corpus from PrivacyConfig

`CorpusBuilder.from_config(...)` wires server-side epsilon/delta from the same config used by the agent.

In [3]:
port = free_port()
base_url = f"http://127.0.0.1:{port}"
sim_proc = launch_sim_server(port)
wait_for_health(base_url, proc=sim_proc)

docs = [
    RawDocument(doc_id="q3", text="Q3 risk report highlights vendor concentration and delayed remediation."),
    RawDocument(doc_id="policy", text="Policy requires quarterly risk treatment tracking and ownership."),
    RawDocument(doc_id="ops", text="Operational incidents increased due to queue saturation in ingestion."),
]

cfg = PrivacyConfig(
    protocol=PrivacyProtocol.DIFF_PRIVACY,
    epsilon=9.0,
    delta=1e-5,
    noise_std=1.0,
    top_k=2,
    max_rounds=4,
    backend=base_url,
)

corpus = (
    CorpusBuilder.from_config(cfg)
    .with_chunk_size(180)
    .add_documents(docs)
    .build_local(workers=2)
)

print("Corpus protocol:", corpus.protocol)
print("Index id:", corpus.index_id)
print("Doc count:", corpus.meta.doc_count)

Corpus protocol: PrivacyProtocol.DIFF_PRIVACY
Index id: 834cfe0f-da7f-43d6-9a12-e84df24294d0
Doc count: 3


## 3) Run a Multi-Round DP Agent and Inspect Budget

Use `OllamaLLM(use_ollama=False)` for deterministic local fallback behavior in tutorials.

In [4]:
agent = SecureRAGAgent.from_config(
    cfg,
    corpus=corpus,
    llm=OllamaLLM(use_ollama=False),
)

result = agent.run("What are the key Q3 risk concerns?")
snapshot = agent.budget_snapshot()

print("Answer preview:", result.answer[:140])
print("Context size:", result.context_size)
print("Rounds used:", result.rounds)
print_snapshot(snapshot)

next_cost = agent.retriever.privacy_cost("another follow-up query")
print("Projected incremental epsilon for one more round:", round(next_cost, 6))

Answer preview: Answer for: What are the key Q3 risk concerns?

Evidence:
- [q3] Q3 risk report highlights vendor concentration and delayed remediation. (sc
Context size: 3
Rounds used: 3
spent: 7.837642
remaining: 1.162358
rounds: 2
mechanism: GaussianMechanism
ledger: [(1, 5.644704), (2, 7.837642)]
Projected incremental epsilon for one more round: 2.0


## 4) Ergonomics Check: Failed Retrieval Does Not Burn Budget

This validates a key developer expectation: if retrieval fails, privacy budget is not debited.

In [5]:
class FailingBackend:
    def embed(self, query: str) -> list[float]:
        return [0.0] * 64

    def retrieve_by_embedding(self, **kwargs):
        raise BackendError("simulated backend failure")


local_cfg = PrivacyConfig(
    protocol=PrivacyProtocol.DIFF_PRIVACY,
    epsilon=5.0,
    delta=1e-5,
    noise_std=1.0,
    # Use a valid backend for retriever construction, then replace with a failing mock.
    backend=base_url,
)
local_corpus = SimpleNamespace(protocol=PrivacyProtocol.DIFF_PRIVACY, index_id="idx")
local_retriever = PrivacyRetriever.from_config(local_cfg, local_corpus)
local_retriever._backend = FailingBackend()

try:
    local_retriever.retrieve("q3 risk", round_n=0)
except BackendError as exc:
    print("Caught expected backend error:", exc)

print_snapshot(local_retriever.budget.snapshot())

Caught expected backend error: simulated backend failure
spent: 0.0
remaining: 5.0
rounds: 0
mechanism: GaussianMechanism
ledger: []


<string>:16: UserWarning: noise_std=1.0 costs epsilon~5.64 per round but epsilon=5.0. Round 1 will be rejected. Increase epsilon or noise_std.


In [6]:
# Optional cleanup checkpoint.
# You can run this now, or skip and run the final section first.
if sim_proc.poll() is None:
    sim_proc.terminate()
    try:
        sim_proc.wait(timeout=2)
    except subprocess.TimeoutExpired:
        sim_proc.kill()
    print("sim_server stopped")
else:
    print("sim_server already stopped")

sim_server stopped


## 5 Prototype a Custom DP Mechanism and Compare

This is the real developer loop: define mechanism class, register it, run the same pipeline, compare outcomes.

The cell below auto-restarts the local sim server if it was already stopped by cleanup.

In [7]:
# Ensure the local backend is alive even if the cleanup cell was run earlier.
if sim_proc.poll() is not None:
    sim_proc = launch_sim_server(port)
    wait_for_health(base_url, proc=sim_proc)

class ShiftedGaussianMechanism(DPMechanismPlugin):
    """Example custom mechanism for prototyping.

    Noise is deterministic per query for reproducibility in experiments.
    RDP accounting uses Gaussian cost as a baseline approximation.
    """

    def noise(self, embedding: list[float], sigma: float, *, query: str = "") -> list[float]:
        seed = int.from_bytes(hashlib.sha256(query.encode("utf-8")).digest()[:8], "little")
        rng = random.Random(seed)
        # Slight mean shift can model mechanism variants for ablation studies.
        return [v + rng.gauss(0.02, sigma) for v in embedding]

    def rdp_cost(self, sigma: float, alpha: float) -> float:
        return alpha / (2.0 * sigma * sigma)


DPMechanismPlugin.register("shifted_gaussian", ShiftedGaussianMechanism())
print("Registered DP mechanisms:", DPMechanismPlugin.registered_names())


def run_once(mech: str) -> dict:
    run_cfg = cfg.for_protocol(
        PrivacyProtocol.DIFF_PRIVACY,
        dp_mechanism=mech,
        epsilon=9.0,
        noise_std=1.0,
        max_rounds=4,
        top_k=2,
    )
    run_corpus = (
        CorpusBuilder.from_config(run_cfg)
        .with_chunk_size(180)
        .add_documents(docs)
        .build_local(workers=2)
    )
    run_agent = SecureRAGAgent.from_config(run_cfg, corpus=run_corpus, llm=OllamaLLM(use_ollama=False))
    run_result = run_agent.run("What are the key Q3 risk concerns?")
    snap = run_agent.budget_snapshot()
    return {
        "mechanism": mech,
        "context_size": run_result.context_size,
        "rounds": snap["rounds"],
        "spent": round(snap["spent"], 6),
        "remaining": round(snap["remaining"], 6),
    }


for mech in ["gaussian", "laplace", "shifted_gaussian"]:
    print(run_once(mech))

# Final cleanup after experiments.
if sim_proc.poll() is None:
    sim_proc.terminate()
    try:
        sim_proc.wait(timeout=2)
    except subprocess.TimeoutExpired:
        sim_proc.kill()
print("sim_server stopped (final cleanup)")

Registered DP mechanisms: ['gaussian', 'laplace', 'shifted_gaussian']
{'mechanism': 'gaussian', 'context_size': 3, 'rounds': 2, 'spent': 7.837642, 'remaining': 1.162358}
{'mechanism': 'laplace', 'context_size': 3, 'rounds': 2, 'spent': 2.327682, 'remaining': 6.672318}
{'mechanism': 'shifted_gaussian', 'context_size': 3, 'rounds': 2, 'spent': 7.837642, 'remaining': 1.162358}
sim_server stopped (final cleanup)


## New API Update: `PrivateQuery.required_budget` + `PrivacyContext` hooks

This framework version supports two key ergonomics updates:

- `PrivateQuery(required_budget=False)` lets you run a retrieval pass without consuming budget.
- `PrivacyContext` hooks now execute during DP retrieval (`encode` and `retrieve`) and can override cost composition safely.

In [ ]:
from securerag.context import PrivacyContext
from securerag.cost import RDPCost
from securerag.models import PrivateQuery

ctx = PrivacyContext(strict=True)

@ctx.register_noise_hook("encode")
def _noise_hook(embedding, config, budget_state):
    return embedding, RDPCost(orders=[2.0, 4.0, 8.0, 16.0, 32.0], values=[0.01] * 5)

@ctx.register_budget_hook("retrieve")
def _budget_hook(docs, config, corpus_budgets):
    return RDPCost(orders=[2.0, 4.0, 8.0, 16.0, 32.0], values=[0.02] * 5)

local_retriever.with_context(ctx)
before = local_retriever.budget.snapshot()["rounds"]

with ctx:
    # No budget charge on this pass.
    _ = local_retriever.retrieve(PrivateQuery(text="q3 risk", required_budget=False), round_n=0)
    # Budget charge on this pass.
    _ = local_retriever.retrieve(PrivateQuery(text="q3 risk", required_budget=True), round_n=1)

after = local_retriever.budget.snapshot()["rounds"]
print("budget rounds delta:", after - before)
print("snapshot:", local_retriever.budget.snapshot())